# Attention Sink Analysis

Detect positions that receive disproportionate attention (sinks),
track their formation, and measure attention concentration.

## Why This Matters

Attention sink reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_sink_analysis import (
    attention_sink_detection, bos_attention_profile,
    sink_formation_trajectory, attention_concentration,
    attention_sink_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Sink Detection

Which positions receive disproportionate attention?

In [ ]:
for layer in range(4):
    result = attention_sink_detection(model, tokens, layer=layer, threshold=0.25)
    print(f"  Layer {layer}: {result['n_sinks']} sinks")
    for h in result['per_head']:
        if h['sinks']:
            positions = [s['position'] for s in h['sinks']]
            print(f"    H{h['head']}: sinks at {positions}")

## BOS Attention Profile

How much attention does the first position receive across layers?

In [ ]:
result = bos_attention_profile(model, tokens, position=0)
for p in result['per_layer']:
    bar = '█' * int(p['mean_bos_attention'] * 40)
    print(f"  Layer {p['layer']}: mean_bos={p['mean_bos_attention']:.4f} {bar}")

## Sink Formation Trajectory

In [ ]:
result = sink_formation_trajectory(model, tokens, position=0)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean={p['mean_attention']:.4f}, max={p['max_attention']:.4f}")

## Attention Concentration (Gini)

In [ ]:
for layer in range(4):
    result = attention_concentration(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_gini={result['mean_gini']:.4f}")
    for p in result['per_head']:
        bar = '█' * int(p['gini'] * 30)
        print(f"    H{p['head']}: gini={p['gini']:.4f} {bar}")

## Summary

In [ ]:
result = attention_sink_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: sinks={p['n_sinks']}, gini={p['mean_gini']:.4f}")